# 🌤️ Weather Assistant with openJiuwen (Interact)

本 notebook 演示如何用 **openJiuwen** 搭建一个支持中断恢复的智能天气助手 Agent：

1. 调用付费 API 前，先询问用户是否确认（单节点交互中断）
2. 并发询问「城市」和「温度单位」（多节点交互中断）
3. 调用外部 API 时可能网络超时 / 限额，异常后重试只重新跑失败节点（异常中断恢复）

全程保持 **session_id** 不变，即可在任何时刻断点续跑。

In [ ]:
# 首次运行需安装依赖
# %pip install jiuwen aiohttp

In [1]:
import uuid, random

from openjiuwen.core.workflow import ComponentComposable
from openjiuwen.core.workflow import End
from openjiuwen.core.workflow import Start
from openjiuwen.core.context_engine.base import Context
from openjiuwen.core.graph.executable import Input, Output
from openjiuwen.core.runtime.interaction.interactive_input import InteractiveInput
from openjiuwen.core.runtime.base import ComponentExecutable
from openjiuwen.core.runtime.runtime import Runtime
from openjiuwen.core.runtime.workflow import WorkflowRuntime
from openjiuwen.core.workflow import Workflow

---
## 1️⃣ 节点定义

In [2]:
class CheckBalanceNode(ComponentExecutable, ComponentComposable):
    """Mock：检查账户余额，如果 < 0.01 USD 则强制让用户确认"""
    async def invoke(self, inputs: Input, runtime: Runtime, context: Context) -> Output:
        balance = random.uniform(0.005, 0.02)   # 随机模拟余额
        print(f"[CheckBalance] 当前余额: ${balance:.4f}")
        if balance < 0.01:
            confirm = await runtime.interact(
                "调用天气 API 需要 $0.01，余额可能不足，是否继续？（Yes/No）"
            )
            if confirm.lower() != "yes":
                raise Exception("用户拒绝付费，流程终止")
        return {"balance_ok": True}

In [3]:
class AskCityNode(ComponentExecutable, ComponentComposable):
    async def invoke(self, inputs: Input, runtime: Runtime, context: Context) -> Output:
        city = await runtime.interact("请输入要查询的城市（如 Beijing）：")
        return {"city": city.strip()}

In [4]:
class AskUnitNode(ComponentExecutable, ComponentComposable):
    async def invoke(self, inputs: Input, runtime: Runtime, context: Context) -> Output:
        unit = await runtime.interact("温度单位选 Celsius 还是 Fahrenheit？（C/F）")
        unit = unit.strip().upper()
        if unit not in {"C", "F"}:
            unit = "C"
        return {"unit": unit}

In [5]:
class QueryWeatherNode(ComponentExecutable, ComponentComposable):
    """Mock：50% 概率抛网络异常，第二次重试一定成功"""
    def __init__(self):
        super().__init__()
        self.tried = False

    async def invoke(self, inputs: Input, runtime: Runtime, context: Context) -> Output:
        city = inputs.get("city")
        unit = inputs.get("unit")
        print(f"[QueryWeather] 开始查询 {city} 单位={unit}")

        # 第一次 50% 概率失败
        if not self.tried and random.random() < 0.5:
            self.tried = True
            raise Exception("网络超时：连接 openweathermap.org 失败")

        # Mock 返回
        temp = random.randint(-10, 35)
        if unit == "F":
            temp = temp * 9 // 5 + 32
        summary = "sunny" if temp > 20 else "cloudy"
        return {"weather": f"{city} 明天 {temp}°{unit} {summary}"}

---
## 2️⃣ 构图

In [6]:
def build_weather_agent():
    flow = Workflow()
    flow.set_start_comp("start", Start({}), inputs_schema={"trigger": "${trigger}"})

    flow.add_workflow_comp("check", CheckBalanceNode())

    flow.add_workflow_comp("ask_city", AskCityNode(),
                           outputs_schema={"city": "${city}"})
    flow.add_workflow_comp("ask_unit", AskUnitNode(),
                           outputs_schema={"unit": "${unit}"})

    flow.add_workflow_comp("weather", QueryWeatherNode(),
                           inputs_schema={"city": "${ask_city.city}", "unit": "${ask_unit.unit}"},
                           outputs_schema={"report": "${weather}"})

    flow.set_end_comp("end", End(),
                      inputs_schema={"report": "${weather.report}"})

    flow.add_connection("start", "check")
    flow.add_connection("check", "ask_city")
    flow.add_connection("check", "ask_unit")
    flow.add_connection("ask_city", "weather")
    flow.add_connection("ask_unit", "weather")
    flow.add_connection("weather", "end")
    return flow

---
## 3️⃣ 运行：第一次可能触发余额确认 + 网络异常

In [7]:
agent = build_weather_agent()
session_id = uuid.uuid4().hex
print("🌀 会话ID =", session_id)

🌀 会话ID = 36ff36faee4347e68333521a071e30bf


In [8]:
# 第一次调用 → 可能被余额检查中断，也可能被网络异常中断
try:
    out = await agent.invoke({"trigger": "明天我要出差，帮我查天气"}, 
                             WorkflowRuntime(session_id=session_id))
    print("☀️ 最终结果：", out.result.get("output").get("report"))
except Exception as e:
    print("💥 第一次执行被中断：", e)

2025-10-13 20:09:06 | common | base.py | 256 | invoke |  | INFO | begin to invoke, input={'trigger': '明天我要出差，帮我查天气'}
[CheckBalance] 当前余额: $0.0163
2025-10-13 20:09:06 | common | manager.py | 42 | stream_output |  | INFO | Received stream data: type='__interaction__' index=0 payload=InteractionOutput(id='ask_city', value='请输入要查询的城市（如 Beijing）：')
2025-10-13 20:09:06 | common | manager.py | 42 | stream_output |  | INFO | Received stream data: type='__interaction__' index=0 payload=InteractionOutput(id='ask_unit', value='温度单位选 Celsius 还是 Fahrenheit？（C/F）')
2025-10-13 20:09:06 | common | manager.py | 37 | stream_output |  | INFO | Received END_FRAME, stopping stream output.
2025-10-13 20:09:06 | common | emitter.py | 67 | close |  | INFO | StreamQueue closed successfully, timeout: 5.0
2025-10-13 20:09:07 | common | base.py | 272 | invoke |  | INFO | end to invoke, results=result=[OutputSchema(type='__interaction__', index=0, payload=InteractionOutput(id='ask_city', value='请输入要查询的城市（如 Beijing

根据中断类型，下面分别处理：

In [9]:
# 如果是交互中断，out.result 里会携带问题
if 'out' in locals() and hasattr(out, 'result') and out.result and isinstance(out.result, list):
    for q in out.result:
        print("❓", q.payload.value)

❓ 请输入要查询的城市（如 Beijing）：
❓ 温度单位选 Celsius 还是 Fahrenheit？（C/F）


In [10]:
# 构造用户回答（余额确认 + 城市 + 单位）
user_input = InteractiveInput()

# 1. 余额确认（若曾被问）
for q in out.result:
    if "余额" in q.payload.value:
        user_input.update(q.payload.id, "Yes")   # 同意付费

# 2. 城市 & 单位（若曾被问）
for q in out.result:
    if "城市" in q.payload.value:
        user_input.update(q.payload.id, "Shanghai")
    elif "温度单位" in q.payload.value:
        user_input.update(q.payload.id, "C")

In [11]:
# 恢复运行（若第一次已 success 则本格直接拿到结果）
final = await agent.invoke(user_input, WorkflowRuntime(session_id=session_id))
print("☀️ 最终结果：", final.result.get("output").get("report"))

2025-10-13 20:09:21 | common | base.py | 256 | invoke |  | INFO | begin to invoke, input=user_inputs={'ask_city': 'Shanghai', 'ask_unit': 'C'} raw_inputs=None
[QueryWeather] 开始查询 Shanghai 单位=C
2025-10-13 20:09:21 | common | manager.py | 37 | stream_output |  | INFO | Received END_FRAME, stopping stream output.
2025-10-13 20:09:21 | common | emitter.py | 67 | close |  | INFO | StreamQueue closed successfully, timeout: 5.0
2025-10-13 20:09:21 | common | base.py | 272 | invoke |  | INFO | end to invoke, results=result={'responseContent': '', 'output': {'report': 'Shanghai 明天 20°C cloudy'}} state=<WorkflowExecutionState.COMPLETED: 'COMPLETED'>
☀️ 最终结果： Shanghai 明天 20°C cloudy


---
## 4️⃣ 再跑一次：余额检查不会再执行（已成功节点跳过）

In [12]:
# 全新会话，观察「余额检查」会再跑一次；但旧会话重试不会跑
new_session = uuid.uuid4().hex
print("🌀 新会话ID =", new_session)

try:
    await agent.invoke({"trigger": "查天气"}, WorkflowRuntime(session_id=new_session))
except Exception as e:
    print("💥 新会话第一次中断：", e)

🌀 新会话ID = 776cfbebf08c47889baf7b8681344162
2025-10-13 20:09:24 | common | base.py | 256 | invoke |  | INFO | begin to invoke, input={'trigger': '查天气'}
[CheckBalance] 当前余额: $0.0197
2025-10-13 20:09:24 | common | manager.py | 42 | stream_output |  | INFO | Received stream data: type='__interaction__' index=0 payload=InteractionOutput(id='ask_city', value='请输入要查询的城市（如 Beijing）：')
2025-10-13 20:09:24 | common | manager.py | 42 | stream_output |  | INFO | Received stream data: type='__interaction__' index=0 payload=InteractionOutput(id='ask_unit', value='温度单位选 Celsius 还是 Fahrenheit？（C/F）')
2025-10-13 20:09:24 | common | manager.py | 37 | stream_output |  | INFO | Received END_FRAME, stopping stream output.
2025-10-13 20:09:24 | common | emitter.py | 67 | close |  | INFO | StreamQueue closed successfully, timeout: 5.0
2025-10-13 20:09:24 | common | base.py | 272 | invoke |  | INFO | end to invoke, results=result=[OutputSchema(type='__interaction__', index=0, payload=InteractionOutput(id='ask

---
## ✅ 总结

| 能力 | 用户视角 | 开发者视角 |
|---|---|---|
| 交互中断 | 弹窗确认 / 输入城市 | 同一 `session_id` + `InteractiveInput` 回复即可续跑 |
| 异常中断 | 网络失败提示重试 | 抛异常后重试，只重跑失败节点，已成功的「余额检查」不再扣费 |
| 多节点并发 | 一次问多个问题 | 为不同 `node_id` 分别 `update()` 回答 |

把这段代码直接搬进项目，你就拥有了一个 **会省钱、会容错、会等人** 的智能天气 Agent！